# Kannolo Sparse Plain Index Benchmark (DotProduct)

## Import Required Libraries

In [ ]:
import importlib.metadata as md

print("kannolo", md.version("kannolo"))

In [ ]:
import kannolo
import numpy as np
import time
from pathlib import Path

## Load and Prepare Sparse Dataset

In [ ]:
# Sparse dataset file in seismic binary format (.bin). Adjust paths as needed.
data_path = ""
print(f"Loading sparse data from: {data_path}")

# Folder with the queries as .npy arrays (see below)
queries_path = ""
print(f"Loading sparse queries from: {queries_path}")

## Create Flat Index and Compute Ground Truth

In [ ]:
# Exhaustive (flat) index for maximum inner product search
flat_index = kannolo.build(data_path, metric="ip", index=kannolo.Flat())
print("Sparse flat index built")

In [ ]:
# Load queries and compute ground truth (k=10)
k = 10

# Read queries from three flat .npy files:
# - components.npy: components of the sparse query vectors, concatenated
# - values.npy: values of the non-zero components, concatenated
# - offsets.npy: where each query starts in components and values (n_queries + 1 entries)
# If your queries are in binary seismic format (.bin), use scripts/convert_bin_to_npy_arrays.py to convert them
components = np.load(Path(queries_path) / "components.npy")
values = np.load(Path(queries_path) / "values.npy")
offsets = np.load(Path(queries_path) / "offsets.npy")
print(f"Loaded {len(offsets)-1} queries from .npy files")
n_queries = len(offsets) - 1

In [ ]:
components.dtype, values.dtype, offsets.dtype

In [ ]:
components.shape, values.shape, offsets.shape

In [ ]:
# A (components, values, offsets) tuple is a batch of queries. kannolo converts the dtypes itself.
queries = (components, values, offsets)

In [ ]:
# Results from flat index search: ids of shape (n_queries, k)
distances_gt, groundtruth = flat_index.search(queries, k)
print(f"Ground truth computed for {n_queries} queries, k={k}")

## Build Sparse HNSW Index

In [ ]:
# HNSW with m=32 neighbors and ef_construction=200, vectors stored with the blocked sparse
# encoding (f16 values)
start = time.time()
hnsw = kannolo.build(data_path, metric="ip", encoder=kannolo.Blocked("f16"), index=kannolo.HNSW(m=32, ef_construction=200))
end = time.time()
print(f"Sparse HNSW index built in {end - start} seconds")
print(hnsw)

## Search with Multiple Parameter Combinations

In [ ]:
# Parameter configurations to test
configs = [
    {"ef_search": 20, "early_exit_threshold": None},
    {"ef_search": 100, "early_exit_threshold": None},
    {"ef_search": 15, "early_exit_threshold": 0.1}
]

results = []

for config in configs:
    ef_search = config["ef_search"]
    early_exit = config["early_exit_threshold"]

    # Search (single-threaded, to time per-query latency)
    start = time.time()
    distances, results_ids = hnsw.search(queries, k, ef_search=ef_search, early_exit_threshold=early_exit, num_threads=1)
    elapsed = time.time() - start

    # Compute accuracy: results and ground truth are (n_queries, k) arrays
    matches = sum(len(set(gt) & set(res)) for gt, res in zip(groundtruth, results_ids))
    accuracy = (matches / (n_queries * k)) * 100
    avg_time = (elapsed / n_queries) * 10**6  # us per query

    results.append({
        "ef_search": ef_search,
        "early_exit_threshold": early_exit,
        "avg_time_us": avg_time,
        "accuracy": accuracy
    })

    print(f"ef_search={ef_search}, early_exit={early_exit}: "
          f"avg_time={avg_time:.3f}us, accuracy={accuracy:.1f}%")

print("\n" + "="*60)
for r in results:
    print(f"Config: ef_search={r['ef_search']}, early_exit={r['early_exit_threshold']}")
    print(f"  Avg Query Time: {r['avg_time_us']:.3f} us")
    print(f"  Accuracy: {r['accuracy']:.1f}%")
    print()